In [1]:
# STEP 1: AUDIT EXISTING DATA AND DATABASE ARCHITECTURE

from pathlib import Path
import pandas as pd
import duckdb

# ------------------------------------------------------------
# PART 1: LOCATE PROJECT ROOT
# ------------------------------------------------------------

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (
            (path / "data" / "processed").is_dir()
            and (
                path / "data" / "warehouse"
                / "demand_intelligence.duckdb"
            ).is_file()
        )
    ),
    None
)

assert project_root is not None, (
    "Could not locate project root and existing database."
)

processed_dir = project_root / "data" / "processed"

database_path = (
    project_root
    / "data"
    / "warehouse"
    / "demand_intelligence.duckdb"
)

print("===== PROJECT LOCATION =====")
print(project_root)

print("\n===== DATABASE LOCATION =====")
print(database_path)


# ------------------------------------------------------------
# PART 2: INVENTORY PROCESSED CSV FILES
# ------------------------------------------------------------

csv_paths = sorted(
    processed_dir.glob("*.csv")
)

processed_names = {
    path.name for path in csv_paths
}

print("\n===== PROCESSED DATA INVENTORY =====")
print("CSV files found:", len(csv_paths))

assert len(csv_paths) >= 37, (
    "Expected at least 37 processed CSV files."
)


# ------------------------------------------------------------
# PART 3: OPEN EXISTING DATABASE IN READ-ONLY MODE
# ------------------------------------------------------------

con = duckdb.connect(
    str(database_path),
    read_only=True
)

try:

    # --------------------------------------------------------
    # PART 4: READ ARCHITECTURE METADATA
    # --------------------------------------------------------

    architecture = con.execute(
        """
        SELECT
            File_Name,
            Source_Module,
            Grain,
            Business_Meaning,
            Analytical_Role
        FROM metadata.architecture_catalog
        ORDER BY File_Name
        """
    ).fetchdf()

    quality = con.execute(
        """
        SELECT *
        FROM metadata.data_quality_report
        """
    ).fetchdf()

    print("\n===== METADATA CATALOG =====")

    print(
        "Architecture records:",
        len(architecture)
    )

    print(
        "Data quality records:",
        len(quality)
    )

    display(
        architecture.groupby("Source_Module")
        .size()
        .reset_index(name="Files")
    )

    assert len(architecture) == 37
    assert len(quality) == 37

    assert architecture["File_Name"].is_unique
    assert quality["File_Name"].is_unique


    # --------------------------------------------------------
    # PART 5: INSPECT DATABASE TABLES
    # --------------------------------------------------------

    database_tables = con.execute(
        """
        SELECT
            table_schema AS Schema,
            table_name AS Table_Name

        FROM information_schema.tables

        WHERE table_schema IN (
            'demand',
            'ecommerce',
            'retail',
            'intent',
            'inventory',
            'sensitivity',
            'metadata'
        )

        AND table_type = 'BASE TABLE'

        ORDER BY table_schema, table_name
        """
    ).fetchdf()

    schema_summary = (
        database_tables.groupby("Schema")
        .size()
        .reset_index(name="Tables")
    )

    print("\n===== DATABASE SCHEMA INVENTORY =====")

    display(schema_summary)

    assert len(database_tables) == 40


    # --------------------------------------------------------
    # PART 6: INSPECT REPORTING VIEWS
    # --------------------------------------------------------

    reporting_views = con.execute(
        """
        SELECT
            table_name AS View_Name,
            view_definition AS View_Definition

        FROM information_schema.views

        WHERE table_schema = 'reporting'

        ORDER BY table_name
        """
    ).fetchdf()

    print("\n===== REPORTING VIEW INVENTORY =====")

    display(
        reporting_views[["View_Name"]]
    )

    assert len(reporting_views) == 16

    assert reporting_views[
        "View_Definition"
    ].notna().all()


    # --------------------------------------------------------
    # PART 7: RECONCILE CSV FILES WITH SQL TABLES
    # --------------------------------------------------------

    catalog_files = set(
        architecture["File_Name"]
    )

    missing_csv_files = (
        catalog_files - processed_names
    )

    additional_csv_files = (
        processed_names - catalog_files
    )

    print("\n===== SOURCE FILE RECONCILIATION =====")

    print(
        "Cataloged CSV files:",
        len(catalog_files)
    )

    print(
        "Missing cataloged files:",
        len(missing_csv_files)
    )

    print(
        "Additional uncataloged files:",
        len(additional_csv_files)
    )

    assert not missing_csv_files, (
        f"Missing files: {missing_csv_files}"
    )


    # --------------------------------------------------------
    # PART 8: RECONCILE EVERY SOURCE TABLE
    # --------------------------------------------------------

    audit_records = []

    for row in architecture.itertuples(index=False):

        filename = row.File_Name

        file_path = processed_dir / filename

        csv_df = pd.read_csv(file_path)

        table_name = Path(filename).stem

        matches = database_tables[
            database_tables["Table_Name"] == table_name
        ]

        assert len(matches) == 1, (
            f"Expected exactly one SQL table for {filename}"
        )

        schema_name = matches["Schema"].iloc[0]

        assert schema_name.isidentifier()
        assert table_name.isidentifier()

        sql_rows = con.execute(
            f"""
            SELECT COUNT(*)
            FROM "{schema_name}"."{table_name}"
            """
        ).fetchone()[0]

        csv_rows = len(csv_df)

        matching_quality = quality[
            quality["File_Name"] == filename
        ]

        assert len(matching_quality) == 1

        documented_rows = int(
            matching_quality["Rows"].iloc[0]
        )

        status = (
            "PASS"
            if (
                csv_rows == sql_rows
                and csv_rows == documented_rows
            )
            else "REVIEW"
        )

        audit_records.append({
            "File_Name": filename,
            "Source_Module": row.Source_Module,
            "Schema": schema_name,
            "Table_Name": table_name,
            "CSV_Rows": csv_rows,
            "SQL_Rows": sql_rows,
            "Documented_Rows": documented_rows,
            "Status": status
        })


    # --------------------------------------------------------
    # PART 9: CREATE RECONCILIATION REPORT
    # --------------------------------------------------------

    pipeline_audit = pd.DataFrame(
        audit_records
    )

    print("\n===== SOURCE-TO-SQL AUDIT =====")

    display(pipeline_audit.head(12))

    print("\n===== AUDIT SUMMARY =====")

    audit_summary = (
        pipeline_audit
        .groupby(["Schema", "Status"])
        .size()
        .reset_index(name="Files")
    )

    display(audit_summary)

    assert len(pipeline_audit) == 37

    assert (
        pipeline_audit["Status"] == "PASS"
    ).all()


    # --------------------------------------------------------
    # PART 10: FINAL ARCHITECTURE VALIDATION
    # --------------------------------------------------------

    assert (
        len(database_tables)
        == len(pipeline_audit) + 3
    )

    assert (
        set(quality["File_Name"])
        == catalog_files
    )

    print("\n===== FINAL PIPELINE AUDIT =====")

    print("37 processed datasets reconciled: PASS")
    print("40 persistent SQL tables: PASS")
    print("16 reporting views: PASS")
    print("37 architecture records: PASS")
    print("37 data quality records: PASS")
    print("CSV-to-SQL row counts: PASS")

    print("\n===== STEP 1 COMPLETE =====")

finally:

    con.close()

    print("\nRead-only database connection closed.")


===== PROJECT LOCATION =====
/Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner

===== DATABASE LOCATION =====
/Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence.duckdb

===== PROCESSED DATA INVENTORY =====
CSV files found: 37

===== METADATA CATALOG =====
Architecture records: 37
Data quality records: 37


,Source_Module,Files
0,Historical Demand Forecasting,8
1,Inventory & Launch Readiness,8
2,Inventory Sensitivity Analysis,8
3,Purchase Intent,2
4,Retail Analytics,7
5,eCommerce Behavior,4



===== DATABASE SCHEMA INVENTORY =====


,Schema,Tables
0,demand,8
1,ecommerce,4
2,intent,2
3,inventory,8
4,metadata,3
5,retail,7
6,sensitivity,8



===== REPORTING VIEW INVENTORY =====


,View_Name
0,v_airpods_weekly_demand
1,v_forecast_lead_time_baseline
2,v_forecast_model_performance
3,v_forecast_multihorizon_performance
4,v_headphone_daily_signals
5,v_inventory_candidate_actions
6,v_inventory_launch_readiness
7,v_inventory_scenario_performance
8,v_inventory_weekly
9,v_purchase_intent_models



===== SOURCE FILE RECONCILIATION =====
Cataloged CSV files: 37
Missing cataloged files: 0
Additional uncataloged files: 0

===== SOURCE-TO-SQL AUDIT =====


,File_Name,Source_Module,Schema,Table_Name,CSV_Rows,SQL_Rows,Documented_Rows,Status
0,airpods_weekly_2019.csv,Historical Demand Forecasting,demand,airpods_weekly_2019,51,51,51,PASS
1,apple_headphones_daily_2019.csv,eCommerce Behavior,ecommerce,apple_headphones_daily_2019,61,61,61,PASS
2,apple_headphones_weekly_2019.csv,eCommerce Behavior,ecommerce,apple_headphones_weekly_2019,9,9,9,PASS
3,demand_sensing_holdout_predictions_2019.csv,eCommerce Behavior,ecommerce,demand_sensing_holdout_predictions_2019,17,17,17,PASS
4,demand_sensing_model_comparison_2019.csv,eCommerce Behavior,ecommerce,demand_sensing_model_comparison_2019,3,3,3,PASS
5,forecast_lead_time_baseline_2019.csv,Historical Demand Forecasting,demand,forecast_lead_time_baseline_2019,4,4,4,PASS
6,forecast_multihorizon_performance_2019.csv,Historical Demand Forecasting,demand,forecast_multihorizon_performance_2019,15,15,15,PASS
7,forecast_multihorizon_predictions_2019.csv,Historical Demand Forecasting,demand,forecast_multihorizon_predictions_2019,240,240,240,PASS
8,forecast_stability_summary_2019.csv,Historical Demand Forecasting,demand,forecast_stability_summary_2019,6,6,6,PASS
9,forecast_uncertainty_summary_2019.csv,Historical Demand Forecasting,demand,forecast_uncertainty_summary_2019,3,3,3,PASS



===== AUDIT SUMMARY =====


,Schema,Status,Files
0,demand,PASS,8
1,ecommerce,PASS,4
2,intent,PASS,2
3,inventory,PASS,8
4,retail,PASS,7
5,sensitivity,PASS,8



===== FINAL PIPELINE AUDIT =====
37 processed datasets reconciled: PASS
40 persistent SQL tables: PASS
16 reporting views: PASS
37 architecture records: PASS
37 data quality records: PASS
CSV-to-SQL row counts: PASS

===== STEP 1 COMPLETE =====

Read-only database connection closed.


In [2]:
# STEP 2: REBUILD SOURCE TABLES IN AN ISOLATED DATABASE

from pathlib import Path
import pandas as pd
import duckdb

# ------------------------------------------------------------
# PART 1: DEFINE SAFE REBUILD LOCATIONS
# ------------------------------------------------------------

config_dir = project_root / "config"
config_dir.mkdir(parents=True, exist_ok=True)

rebuild_path = (
    project_root
    / "data"
    / "warehouse"
    / "demand_intelligence_rebuild_candidate.duckdb"
)

assert rebuild_path.resolve() != database_path.resolve()

print("===== REBUILD CONFIGURATION =====")
print("Original database:", database_path)
print("Rebuild candidate:", rebuild_path)

# Quote identifiers safely for SQL.
def quote_identifier(value):
    return '"' + str(value).replace('"', '""') + '"'


# ------------------------------------------------------------
# PART 2: CREATE SOURCE TABLE MANIFEST
# ------------------------------------------------------------

table_manifest = (
    pipeline_audit[
        ["File_Name", "Schema", "Table_Name", "CSV_Rows"]
    ]
    .sort_values(["Schema", "Table_Name"])
    .reset_index(drop=True)
    .copy()
)

assert len(table_manifest) == 37
assert table_manifest["File_Name"].is_unique

manifest_path = config_dir / "table_manifest.csv"

table_manifest.to_csv(
    manifest_path,
    index=False
)

print("\n===== TABLE MANIFEST =====")
print("Source tables documented:", len(table_manifest))

display(
    table_manifest.groupby("Schema")
    .size()
    .reset_index(name="Tables")
)


# ------------------------------------------------------------
# PART 3: CAPTURE ORIGINAL SQL COLUMN DEFINITIONS
# ------------------------------------------------------------

reference_con = duckdb.connect(
    str(database_path),
    read_only=True
)

try:
    column_manifest = reference_con.execute(
        """
        SELECT
            table_schema AS Schema,
            table_name AS Table_Name,
            ordinal_position AS Column_Position,
            column_name AS Column_Name,
            data_type AS Data_Type

        FROM information_schema.columns

        WHERE table_schema IN (
            'demand',
            'ecommerce',
            'retail',
            'intent',
            'inventory',
            'sensitivity'
        )

        ORDER BY
            table_schema,
            table_name,
            ordinal_position
        """
    ).fetchdf()

finally:
    reference_con.close()

column_path = config_dir / "column_schema_manifest.csv"

column_manifest.to_csv(
    column_path,
    index=False
)

assert not column_manifest.empty

print("\n===== COLUMN SCHEMA MANIFEST =====")
print("Column definitions:", len(column_manifest))

display(column_manifest.head(12))


# ------------------------------------------------------------
# PART 4: VERIFY MANIFEST COMPLETENESS
# ------------------------------------------------------------

source_table_keys = set(
    zip(
        table_manifest["Schema"],
        table_manifest["Table_Name"]
    )
)

column_table_keys = set(
    zip(
        column_manifest["Schema"],
        column_manifest["Table_Name"]
    )
)

assert source_table_keys == column_table_keys

print("\n===== MANIFEST VALIDATION =====")
print("All 37 source tables have column definitions: PASS")


# ------------------------------------------------------------
# PART 5: INITIALIZE THE ISOLATED REBUILD
# ------------------------------------------------------------

rebuild_con = duckdb.connect(str(rebuild_path))
transaction_started = False

try:

    rebuild_con.execute("BEGIN TRANSACTION").fetchall()
    transaction_started = True

    # Clear only the dedicated rebuild candidate.
    # The original database is never modified.

    schemas_to_reset = [
        "reporting",
        "metadata",
        "demand",
        "ecommerce",
        "retail",
        "intent",
        "inventory",
        "sensitivity"
    ]

    for schema in schemas_to_reset:
        rebuild_con.execute(
            f"DROP SCHEMA IF EXISTS "
            f"{quote_identifier(schema)} CASCADE"
        ).fetchall()

    source_schemas = sorted(
        table_manifest["Schema"].unique()
    )

    for schema in source_schemas:
        rebuild_con.execute(
            f"CREATE SCHEMA {quote_identifier(schema)}"
        ).fetchall()

    print("\n===== REBUILDING SOURCE TABLES =====")

    rebuild_records = []

    # --------------------------------------------------------
    # PART 6: RECONSTRUCT ALL 37 SOURCE TABLES
    # --------------------------------------------------------

    for source in table_manifest.itertuples(index=False):

        file_path = processed_dir / source.File_Name

        assert file_path.is_file()

        df = pd.read_csv(file_path)

        assert len(df) == int(source.CSV_Rows)

        # Recover the SQL column types saved from
        # the original validated database.

        specs = (
            column_manifest[
                (column_manifest["Schema"] == source.Schema)
                &
                (
                    column_manifest["Table_Name"]
                    == source.Table_Name
                )
            ]
            .sort_values("Column_Position")
        )

        expected_columns = specs["Column_Name"].tolist()

        assert df.columns.tolist() == expected_columns, (
            f"Column mismatch: {source.File_Name}"
        )

        cast_expressions = []

        for col in specs.itertuples(index=False):

            safe_name = quote_identifier(col.Column_Name)

            cast_expressions.append(
                f"CAST({safe_name} AS {col.Data_Type}) "
                f"AS {safe_name}"
            )

        selection = ", ".join(cast_expressions)

        target = (
            f"{quote_identifier(source.Schema)}."
            f"{quote_identifier(source.Table_Name)}"
        )

        rebuild_con.register("incoming_csv", df)

        try:
            rebuild_con.execute(
                f"""
                CREATE TABLE {target} AS
                SELECT {selection}
                FROM incoming_csv
                """
            ).fetchall()
        finally:
            rebuild_con.unregister("incoming_csv")

        actual_rows = rebuild_con.execute(
            f"SELECT COUNT(*) FROM {target}"
        ).fetchone()[0]

        assert actual_rows == len(df)

        # Verify SQL column names and data types.
        actual_specs = rebuild_con.execute(
            """
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = ?
              AND table_name = ?
            ORDER BY ordinal_position
            """,
            [source.Schema, source.Table_Name]
        ).fetchall()

        expected_specs = list(
            zip(
                specs["Column_Name"],
                specs["Data_Type"]
            )
        )

        assert actual_specs == expected_specs, (
            f"SQL column type mismatch: {target}"
        )

        rebuild_records.append({
            "Schema": source.Schema,
            "Table": source.Table_Name,
            "CSV_Rows": len(df),
            "Rebuilt_Rows": actual_rows,
            "Schema_Match": True,
            "Status": "PASS"
        })

        print(
            f"Rebuilt: {source.Schema}."
            f"{source.Table_Name} "
            f"({actual_rows} rows)"
        )


    # --------------------------------------------------------
    # PART 7: VALIDATE REBUILT DATABASE
    # --------------------------------------------------------

    rebuild_report = pd.DataFrame(rebuild_records)

    assert len(rebuild_report) == 37

    assert (rebuild_report["Status"] == "PASS").all()

    rebuilt_table_count = rebuild_con.execute(
        """
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema IN (
            'demand',
            'ecommerce',
            'retail',
            'intent',
            'inventory',
            'sensitivity'
        )
        AND table_type = 'BASE TABLE'
        """
    ).fetchone()[0]

    assert rebuilt_table_count == 37

    rebuild_con.execute("COMMIT").fetchall()
    transaction_started = False

    print("\n===== REBUILD VALIDATION REPORT =====")

    display(
        rebuild_report.groupby(["Schema", "Status"])
        .size()
        .reset_index(name="Tables")
    )

    print("\n===== FINAL STEP 2 VALIDATION =====")

    print("Source CSV files: 37")
    print("Rebuilt SQL tables:", rebuilt_table_count)
    print("CSV-to-SQL row counts: PASS")
    print("SQL column names and types: PASS")
    print("Original database left unchanged: PASS")
    print("Isolated candidate rebuild: PASS")

    print("\n===== STEP 2 COMPLETE =====")

except Exception:
    if transaction_started:
        rebuild_con.execute("ROLLBACK").fetchall()
    raise

finally:
    rebuild_con.close()
    print("\nRebuild database connection closed safely.")


===== REBUILD CONFIGURATION =====
Original database: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence.duckdb
Rebuild candidate: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence_rebuild_candidate.duckdb

===== TABLE MANIFEST =====
Source tables documented: 37


,Schema,Tables
0,demand,8
1,ecommerce,4
2,intent,2
3,inventory,8
4,retail,7
5,sensitivity,8



===== COLUMN SCHEMA MANIFEST =====
Column definitions: 364


,Schema,Table_Name,Column_Position,Column_Name,Data_Type
0,demand,airpods_weekly_2019,1,Week_Ending,VARCHAR
1,demand,airpods_weekly_2019,2,Units_Sold,BIGINT
2,demand,forecast_lead_time_baseline_2019,1,Forecast_Origin,VARCHAR
3,demand,forecast_lead_time_baseline_2019,2,Forecast_Week,VARCHAR
4,demand,forecast_lead_time_baseline_2019,3,Lead_Time_Weeks,BIGINT
5,demand,forecast_lead_time_baseline_2019,4,Model,VARCHAR
6,demand,forecast_lead_time_baseline_2019,5,Forecast_Weekly_Units,DOUBLE
7,demand,forecast_lead_time_baseline_2019,6,Cumulative_Forecast_Units,DOUBLE
8,demand,forecast_lead_time_baseline_2019,7,Scenario_Type,VARCHAR
9,demand,forecast_multihorizon_performance_2019,1,Horizon_Weeks,BIGINT



===== MANIFEST VALIDATION =====
All 37 source tables have column definitions: PASS

===== REBUILDING SOURCE TABLES =====
Rebuilt: demand.airpods_weekly_2019 (51 rows)
Rebuilt: demand.forecast_lead_time_baseline_2019 (4 rows)
Rebuilt: demand.forecast_multihorizon_performance_2019 (15 rows)
Rebuilt: demand.forecast_multihorizon_predictions_2019 (240 rows)
Rebuilt: demand.forecast_stability_summary_2019 (6 rows)
Rebuilt: demand.forecast_uncertainty_summary_2019 (3 rows)
Rebuilt: demand.model_performance_rolling_2019 (5 rows)
Rebuilt: demand.model_predictions_rolling_2019 (19 rows)
Rebuilt: ecommerce.apple_headphones_daily_2019 (61 rows)
Rebuilt: ecommerce.apple_headphones_weekly_2019 (9 rows)
Rebuilt: ecommerce.demand_sensing_holdout_predictions_2019 (17 rows)
Rebuilt: ecommerce.demand_sensing_model_comparison_2019 (3 rows)
Rebuilt: intent.purchase_intent_confusion_summary (2 rows)
Rebuilt: intent.purchase_intent_model_leaderboard (7 rows)
Rebuilt: inventory.inventory_candidate_actions (

,Schema,Status,Tables
0,demand,PASS,8
1,ecommerce,PASS,4
2,intent,PASS,2
3,inventory,PASS,8
4,retail,PASS,7
5,sensitivity,PASS,8



===== FINAL STEP 2 VALIDATION =====
Source CSV files: 37
Rebuilt SQL tables: 37
CSV-to-SQL row counts: PASS
SQL column names and types: PASS
Original database left unchanged: PASS
Isolated candidate rebuild: PASS

===== STEP 2 COMPLETE =====

Rebuild database connection closed safely.


In [3]:
# STEP 3: RESTORE METADATA AND REPORTING VIEWS

from pathlib import Path
import pandas as pd
import duckdb
import re
from collections import Counter

# ------------------------------------------------------------
# PART 1: DEFINE METADATA AND VIEW EXPECTATIONS
# ------------------------------------------------------------

metadata_expectations = {
    "architecture_catalog": 37,
    "data_quality_report": 37,
    "kpi_dictionary": 8
}

expected_reporting_views = 16

config_dir = project_root / "config"
config_dir.mkdir(parents=True, exist_ok=True)

print("===== RESTORATION CONFIGURATION =====")
print("Original database:", database_path)
print("Rebuild candidate:", rebuild_path)


# ------------------------------------------------------------
# PART 2: READ METADATA AND VIEW DEFINITIONS
# ------------------------------------------------------------

reference_con = duckdb.connect(
    str(database_path),
    read_only=True
)

try:

    metadata_snapshots = {}

    for table_name in metadata_expectations:

        metadata_snapshots[table_name] = (
            reference_con.execute(
                f"""
                SELECT *
                FROM metadata.{quote_identifier(table_name)}
                """
            ).fetchdf()
        )

    metadata_columns = reference_con.execute(
        """
        SELECT
            table_name,
            ordinal_position,
            column_name,
            data_type

        FROM information_schema.columns
        WHERE table_schema = 'metadata'

        ORDER BY table_name, ordinal_position
        """
    ).fetchdf()

    view_definitions = reference_con.execute(
        """
        SELECT
            view_name,
            sql

        FROM duckdb_views()
        WHERE schema_name = 'reporting'
        ORDER BY view_name
        """
    ).fetchdf()

    reference_view_counts = {}

    for view_name in view_definitions["view_name"]:

        reference_view_counts[view_name] = (
            reference_con.execute(
                f"""
                SELECT COUNT(*)
                FROM reporting.{quote_identifier(view_name)}
                """
            ).fetchone()[0]
        )

finally:

    reference_con.close()

assert len(view_definitions) == expected_reporting_views

assert (
    len(metadata_snapshots)
    == len(metadata_expectations)
)


# ------------------------------------------------------------
# PART 3: SAVE REBUILD CONFIGURATION FILES
# ------------------------------------------------------------

for table_name, df in metadata_snapshots.items():

    assert len(df) == metadata_expectations[table_name]

    output_path = (
        config_dir / f"metadata_{table_name}.csv"
    )

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved metadata snapshot: {output_path.name}"
    )

metadata_columns.to_csv(
    config_dir / "metadata_column_manifest.csv",
    index=False
)

view_definitions.to_csv(
    config_dir / "reporting_view_manifest.csv",
    index=False
)

print("\n===== CONFIGURATION SNAPSHOTS =====")

print("Metadata snapshots:", len(metadata_snapshots))
print("Metadata schema manifest: 1")
print("Reporting view manifest: 1")


# ------------------------------------------------------------
# PART 4: PREPARE DATA COMPARISON HELPER
# ------------------------------------------------------------

def normalized_records(df):

    records = []

    for row in df.itertuples(
        index=False,
        name=None
    ):

        normalized_row = tuple(
            "<NULL>" if pd.isna(value)
            else str(value)
            for value in row
        )

        records.append(normalized_row)

    return Counter(records)


# ------------------------------------------------------------
# PART 5: CONNECT TO ISOLATED REBUILD DATABASE
# ------------------------------------------------------------

candidate_con = duckdb.connect(str(rebuild_path))

transaction_open = False

try:

    candidate_con.execute(
        "BEGIN TRANSACTION"
    ).fetchall()

    transaction_open = True

    # Reset only the candidate's reporting schema.
    # This also makes this cell rerunnable.

    candidate_con.execute(
        "DROP SCHEMA IF EXISTS reporting CASCADE"
    ).fetchall()

    candidate_con.execute(
        "CREATE SCHEMA reporting"
    ).fetchall()

    candidate_con.execute(
        "CREATE SCHEMA IF NOT EXISTS metadata"
    ).fetchall()


    # --------------------------------------------------------
    # PART 6: RESTORE ALL THREE METADATA TABLES
    # --------------------------------------------------------

    metadata_records = []

    print("\n===== RESTORING METADATA TABLES =====")

    for table_name, df in metadata_snapshots.items():

        column_specs = (
            metadata_columns[
                metadata_columns["table_name"]
                == table_name
            ]
            .sort_values("ordinal_position")
        )

        assert (
            df.columns.tolist()
            == column_specs["column_name"].tolist()
        )

        cast_expressions = []

        for column in column_specs.itertuples(
            index=False
        ):

            safe_column = quote_identifier(
                column.column_name
            )

            cast_expressions.append(
                f"CAST({safe_column} "
                f"AS {column.data_type}) "
                f"AS {safe_column}"
            )

        column_sql = ", ".join(cast_expressions)

        candidate_con.register(
            "incoming_metadata",
            df
        )

        try:

            candidate_con.execute(
                f"""
                CREATE OR REPLACE TABLE
                    metadata.{quote_identifier(table_name)}
                AS
                SELECT {column_sql}
                FROM incoming_metadata
                """
            ).fetchall()

        finally:

            candidate_con.unregister(
                "incoming_metadata"
            )

        restored_df = candidate_con.execute(
            f"""
            SELECT *
            FROM metadata.{quote_identifier(table_name)}
            """
        ).fetchdf()

        assert len(restored_df) == len(df)

        assert (
            normalized_records(restored_df)
            == normalized_records(df)
        ), f"Metadata value mismatch: {table_name}"

        actual_specs = candidate_con.execute(
            """
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'metadata'
              AND table_name = ?
            ORDER BY ordinal_position
            """,
            [table_name]
        ).fetchall()

        expected_specs = list(
            zip(
                column_specs["column_name"],
                column_specs["data_type"]
            )
        )

        assert actual_specs == expected_specs

        metadata_records.append({
            "Table": f"metadata.{table_name}",
            "Expected_Rows": len(df),
            "Actual_Rows": len(restored_df),
            "Status": "PASS"
        })

        print(
            f"Restored: metadata.{table_name} "
            f"({len(restored_df)} rows)"
        )


    # --------------------------------------------------------
    # PART 7: RECREATE ALL 16 REPORTING VIEWS
    # --------------------------------------------------------

    # DuckDB stores complete CREATE VIEW statements.
    # We extract the SELECT definition and recreate
    # it against the isolated candidate's schemas.

    create_view_pattern = re.compile(
        r"^\s*CREATE\s+"
        r"(?:OR\s+REPLACE\s+)?"
        r"(?:TEMPORARY\s+)?"
        r"VIEW\s+"
        r"(?:IF\s+NOT\s+EXISTS\s+)?"
        r".+?\s+AS\s+",
        flags=re.IGNORECASE | re.DOTALL
    )

    parsed_views = {}

    for row in view_definitions.itertuples(index=False):

        stored_sql = str(row.sql).strip()

        match = create_view_pattern.match(
            stored_sql
        )

        assert match is not None, (
            f"Cannot parse view SQL: {row.view_name}"
        )

        select_body = (
            stored_sql[match.end():]
            .strip()
            .rstrip(";")
        )

        assert select_body

        parsed_views[row.view_name] = select_body


    print("\n===== RESTORING REPORTING VIEWS =====")

    # Retry unresolved views in case any view references
    # another reporting view that has not yet been created.

    pending_views = dict(parsed_views)

    for attempt in range(expected_reporting_views):

        if not pending_views:
            break

        remaining = {}
        progress = False

        for view_name, select_sql in (
            pending_views.items()
        ):

            try:

                candidate_con.execute(
                    f"""
                    CREATE OR REPLACE VIEW
                        reporting.{quote_identifier(view_name)}
                    AS
                    {select_sql}
                    """
                ).fetchall()

                progress = True

                print("Created:", view_name)

            except duckdb.Error as error:

                remaining[view_name] = (
                    select_sql,
                    str(error)
                )

        if not remaining:

            pending_views = {}
            break

        if not progress:

            details = "\n".join(
                f"{name}: {error}"
                for name, (_, error) in remaining.items()
            )

            raise RuntimeError(
                "Could not restore reporting views:\n"
                + details
            )

        pending_views = {
            name: sql
            for name, (sql, _) in remaining.items()
        }

    assert not pending_views


    # --------------------------------------------------------
    # PART 8: VALIDATE REPORTING VIEW ROW COUNTS
    # --------------------------------------------------------

    view_records = []

    for view_name, expected in (
        reference_view_counts.items()
    ):

        actual = candidate_con.execute(
            f"""
            SELECT COUNT(*)
            FROM reporting.{quote_identifier(view_name)}
            """
        ).fetchone()[0]

        assert actual == expected, (
            f"Reporting view mismatch: {view_name}"
        )

        view_records.append({
            "View": view_name,
            "Original_Rows": expected,
            "Rebuilt_Rows": actual,
            "Status": "PASS"
        })

    view_report = pd.DataFrame(view_records)

    print("\n===== REPORTING VIEW VALIDATION =====")

    display(view_report)


    # --------------------------------------------------------
    # PART 9: VALIDATE COMPLETE REBUILD STRUCTURE
    # --------------------------------------------------------

    total_tables = candidate_con.execute(
        """
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema IN (
            'demand',
            'ecommerce',
            'retail',
            'intent',
            'inventory',
            'sensitivity',
            'metadata'
        )
        AND table_type = 'BASE TABLE'
        """
    ).fetchone()[0]

    total_views = candidate_con.execute(
        """
        SELECT COUNT(*)
        FROM information_schema.views
        WHERE table_schema = 'reporting'
        """
    ).fetchone()[0]

    assert total_tables == 40
    assert total_views == 16

    assert len(metadata_records) == 3

    assert (
        view_report["Status"] == "PASS"
    ).all()


    # --------------------------------------------------------
    # PART 10: COMMIT SUCCESSFUL REBUILD
    # --------------------------------------------------------

    candidate_con.execute(
        "COMMIT"
    ).fetchall()

    transaction_open = False

    print("\n===== FINAL REBUILD VALIDATION =====")

    print("Analytical source tables: 37")
    print("Metadata tables restored: 3")
    print("Persistent SQL tables:", total_tables)
    print("Reporting views:", total_views)
    print("Metadata row/value checks: PASS")
    print("Reporting view row counts: PASS")
    print("Original database remains unchanged: PASS")

    print("\n===== STEP 3 COMPLETE =====")

except Exception:

    if transaction_open:

        candidate_con.execute(
            "ROLLBACK"
        ).fetchall()

    raise

finally:

    candidate_con.close()

    print("\nCandidate database connection closed safely.")


===== RESTORATION CONFIGURATION =====
Original database: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence.duckdb
Rebuild candidate: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence_rebuild_candidate.duckdb
Saved metadata snapshot: metadata_architecture_catalog.csv
Saved metadata snapshot: metadata_data_quality_report.csv
Saved metadata snapshot: metadata_kpi_dictionary.csv

===== CONFIGURATION SNAPSHOTS =====
Metadata snapshots: 3
Metadata schema manifest: 1
Reporting view manifest: 1

===== RESTORING METADATA TABLES =====
Restored: metadata.architecture_catalog (37 rows)
Restored: metadata.data_quality_report (37 rows)
Restored: metadata.kpi_dictionary (8 rows)

===== RESTORING REPORTING VIEWS =====
Created: v_airpods_weekly_demand
Created: v_forecast_lead_time_baseline
Created: v_forecast_model_performance
Created: v_forecast_multihorizon_performance

,View,Original_Rows,Rebuilt_Rows,Status
0,v_airpods_weekly_demand,51,51,PASS
1,v_forecast_lead_time_baseline,4,4,PASS
2,v_forecast_model_performance,8,8,PASS
3,v_forecast_multihorizon_performance,15,15,PASS
4,v_headphone_daily_signals,61,61,PASS
5,v_inventory_candidate_actions,2,2,PASS
6,v_inventory_launch_readiness,7,7,PASS
7,v_inventory_scenario_performance,3,3,PASS
8,v_inventory_weekly,12,12,PASS
9,v_purchase_intent_models,7,7,PASS



===== FINAL REBUILD VALIDATION =====
Analytical source tables: 37
Metadata tables restored: 3
Persistent SQL tables: 40
Reporting views: 16
Metadata row/value checks: PASS
Reporting view row counts: PASS
Original database remains unchanged: PASS

===== STEP 3 COMPLETE =====

Candidate database connection closed safely.


In [4]:
# STEP 4: VALIDATE ORIGINAL VS REBUILT DATABASE VALUES

from pathlib import Path
import pandas as pd
import duckdb

# ------------------------------------------------------------
# PART 1: VERIFY BOTH DATABASES EXIST
# ------------------------------------------------------------

assert database_path.is_file()
assert rebuild_path.is_file()
assert database_path.resolve() != rebuild_path.resolve()

print("===== DATABASE COMPARISON CONFIGURATION =====")
print("Original:", database_path)
print("Rebuilt:", rebuild_path)


# ------------------------------------------------------------
# PART 2: DEFINE SAFE SQL IDENTIFIERS
# ------------------------------------------------------------

def quote_sql_identifier(value):
    return '"' + str(value).replace('"', '""') + '"'

def quote_sql_path(path):
    return "'" + str(path).replace("'", "''") + "'"


# ------------------------------------------------------------
# PART 3: ATTACH BOTH DATABASES AS READ-ONLY
# ------------------------------------------------------------

compare_con = duckdb.connect(":memory:")

try:

    compare_con.execute(
        f"""
        ATTACH {quote_sql_path(database_path)}
        AS original_db (READ_ONLY)
        """
    ).fetchall()

    compare_con.execute(
        f"""
        ATTACH {quote_sql_path(rebuild_path)}
        AS rebuilt_db (READ_ONLY)
        """
    ).fetchall()

    print("\n===== DATABASE CONNECTIONS =====")
    print("Original attached read-only: PASS")
    print("Rebuilt attached read-only: PASS")


    # --------------------------------------------------------
    # PART 4: BUILD COMPLETE OBJECT INVENTORY
    # --------------------------------------------------------

    objects_to_compare = []

    for row in database_tables.itertuples(index=False):

        objects_to_compare.append({
            "Object_Type": "TABLE",
            "Schema": row.Schema,
            "Object_Name": row.Table_Name
        })

    for view_name in reporting_views["View_Name"]:

        objects_to_compare.append({
            "Object_Type": "VIEW",
            "Schema": "reporting",
            "Object_Name": view_name
        })

    assert len(objects_to_compare) == 56

    print("\n===== COMPARISON SCOPE =====")
    print("SQL tables:", 40)
    print("Reporting views:", 16)
    print("Total SQL objects:", len(objects_to_compare))


    # --------------------------------------------------------
    # PART 5: DEFINE SQL COLUMN COMPARISON
    # --------------------------------------------------------

    def qualified_object(database, schema, name):

        return ".".join([
            quote_sql_identifier(database),
            quote_sql_identifier(schema),
            quote_sql_identifier(name)
        ])

    def describe_object(qualified_name):

        result = compare_con.execute(
            f"DESCRIBE SELECT * FROM {qualified_name}"
        ).fetchall()

        return [
            (row[0], str(row[1]).upper())
            for row in result
        ]

    def normalized_column_expressions(columns):

        expressions = []

        for column_name, data_type in columns:

            safe_column = quote_sql_identifier(
                column_name
            )

            if (
                data_type in ("DOUBLE", "FLOAT", "REAL")
                or data_type.startswith("FLOAT")
            ):

                expression = (
                    f"ROUND({safe_column}, 6) "
                    f"AS {safe_column}"
                )

            else:
                expression = safe_column

            expressions.append(expression)

        return ", ".join(expressions)


    # --------------------------------------------------------
    # PART 6: COMPARE ALL TABLES AND VIEWS
    # --------------------------------------------------------

    parity_records = []

    print("\n===== DATA VALUE PARITY CHECKS =====")

    for obj in objects_to_compare:

        object_type = obj["Object_Type"]
        schema = obj["Schema"]
        name = obj["Object_Name"]

        original = qualified_object(
            "original_db", schema, name
        )

        rebuilt = qualified_object(
            "rebuilt_db", schema, name
        )

        original_columns = describe_object(original)
        rebuilt_columns = describe_object(rebuilt)

        schema_match = (
            original_columns == rebuilt_columns
        )

        original_count = compare_con.execute(
            f"SELECT COUNT(*) FROM {original}"
        ).fetchone()[0]

        rebuilt_count = compare_con.execute(
            f"SELECT COUNT(*) FROM {rebuilt}"
        ).fetchone()[0]

        row_count_match = (
            original_count == rebuilt_count
        )

        original_only = None
        rebuilt_only = None

        if schema_match:

            projection = normalized_column_expressions(
                original_columns
            )

            # EXCEPT ALL preserves duplicate-row
            # multiplicities and matches NULL values.

            original_only = compare_con.execute(
                f"""
                SELECT COUNT(*)
                FROM (
                    SELECT {projection}
                    FROM {original}

                    EXCEPT ALL

                    SELECT {projection}
                    FROM {rebuilt}
                ) AS difference_rows
                """
            ).fetchone()[0]

            rebuilt_only = compare_con.execute(
                f"""
                SELECT COUNT(*)
                FROM (
                    SELECT {projection}
                    FROM {rebuilt}

                    EXCEPT ALL

                    SELECT {projection}
                    FROM {original}
                ) AS difference_rows
                """
            ).fetchone()[0]

        values_match = (
            schema_match
            and original_only == 0
            and rebuilt_only == 0
        )

        status = (
            "PASS"
            if (
                schema_match
                and row_count_match
                and values_match
            )
            else "REVIEW"
        )

        parity_records.append({
            "Object_Type": object_type,
            "Schema": schema,
            "Object_Name": name,
            "Original_Rows": original_count,
            "Rebuilt_Rows": rebuilt_count,
            "Schema_Match": schema_match,
            "Original_Only_Rows": original_only,
            "Rebuilt_Only_Rows": rebuilt_only,
            "Status": status
        })

        print(
            f"{status}: {schema}.{name}"
        )


    # --------------------------------------------------------
    # PART 7: BUILD VALIDATION REPORT
    # --------------------------------------------------------

    parity_report = pd.DataFrame(
        parity_records
    )

    print("\n===== DATA PARITY VALIDATION SUMMARY =====")

    parity_summary = (
        parity_report
        .groupby(["Object_Type", "Status"])
        .size()
        .reset_index(name="Objects")
    )

    display(parity_summary)

    failed_objects = parity_report[
        parity_report["Status"] != "PASS"
    ]

    print("\n===== OBJECTS REQUIRING REVIEW =====")

    if failed_objects.empty:
        print("None. All objects passed.")
    else:
        display(failed_objects)


    # --------------------------------------------------------
    # PART 8: VALIDATE BUSINESS QUERY RESULTS
    # --------------------------------------------------------

    business_checks = [
        {
            "Metric": "Historical AirPods units",
            "Query": """
                SELECT SUM(Units_Sold)
                FROM {db}.reporting.v_airpods_weekly_demand
            """,
            "Expected": 15364
        },
        {
            "Metric": "Supplier-delay shortage units",
            "Query": """
                SELECT SUM(Shortage_Units)
                FROM {db}.reporting.v_inventory_weekly
                WHERE Scenario = 'Supplier Delay'
            """,
            "Expected": 660
        },
        {
            "Metric": "No READY option cases",
            "Query": """
                SELECT COUNT(*)
                FROM {db}.reporting.v_sensitivity_decision_cases
                WHERE Lowest_Ready_Action = 'NO READY OPTION'
                  AND Lowest_Ready_Exposure_USD IS NULL
            """,
            "Expected": 72
        }
    ]

    business_records = []

    for check in business_checks:

        original_sql = check["Query"].format(
            db=quote_sql_identifier("original_db")
        )

        rebuilt_sql = check["Query"].format(
            db=quote_sql_identifier("rebuilt_db")
        )

        original_value = compare_con.execute(
            original_sql
        ).fetchone()[0]

        rebuilt_value = compare_con.execute(
            rebuilt_sql
        ).fetchone()[0]

        passed = (
            original_value == rebuilt_value
            and rebuilt_value == check["Expected"]
        )

        business_records.append({
            "Metric": check["Metric"],
            "Original": original_value,
            "Rebuilt": rebuilt_value,
            "Expected": check["Expected"],
            "Status": "PASS" if passed else "REVIEW"
        })

    business_report = pd.DataFrame(
        business_records
    )

    print("\n===== BUSINESS QUERY RECONCILIATION =====")

    display(business_report)


    # --------------------------------------------------------
    # PART 9: FINAL VALIDATION
    # --------------------------------------------------------

    assert len(parity_report) == 56

    assert len(
        parity_report[
            parity_report["Object_Type"] == "TABLE"
        ]
    ) == 40

    assert len(
        parity_report[
            parity_report["Object_Type"] == "VIEW"
        ]
    ) == 16

    assert failed_objects.empty, (
        "One or more database objects require review."
    )

    assert (
        business_report["Status"] == "PASS"
    ).all()

    print("\n===== FINAL END-TO-END VALIDATION =====")

    print("40 table comparisons: PASS")
    print("16 reporting view comparisons: PASS")
    print("SQL column names and types: PASS")
    print("Row-count reconciliation: PASS")
    print("Normalized data-value reconciliation: PASS")
    print("Business query reconciliation: PASS")
    print("Original database protected: PASS")

    print("\n===== STEP 4 COMPLETE =====")

finally:

    compare_con.close()

    print("\nComparison connection closed safely.")


===== DATABASE COMPARISON CONFIGURATION =====
Original: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence.duckdb
Rebuilt: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence_rebuild_candidate.duckdb

===== DATABASE CONNECTIONS =====
Original attached read-only: PASS
Rebuilt attached read-only: PASS

===== COMPARISON SCOPE =====
SQL tables: 40
Reporting views: 16
Total SQL objects: 56

===== DATA VALUE PARITY CHECKS =====
PASS: demand.airpods_weekly_2019
PASS: demand.forecast_lead_time_baseline_2019
PASS: demand.forecast_multihorizon_performance_2019
PASS: demand.forecast_multihorizon_predictions_2019
PASS: demand.forecast_stability_summary_2019
PASS: demand.forecast_uncertainty_summary_2019
PASS: demand.model_performance_rolling_2019
PASS: demand.model_predictions_rolling_2019
PASS: ecommerce.apple_headphones_daily_2019
PASS: ecommerce.apple_headphones_wee

,Object_Type,Status,Objects
0,TABLE,PASS,40
1,VIEW,PASS,16



===== OBJECTS REQUIRING REVIEW =====
None. All objects passed.

===== BUSINESS QUERY RECONCILIATION =====


,Metric,Original,Rebuilt,Expected,Status
0,Historical AirPods units,15364.0,15364.0,15364,PASS
1,Supplier-delay shortage units,660.0,660.0,660,PASS
2,No READY option cases,72.0,72.0,72,PASS



===== FINAL END-TO-END VALIDATION =====
40 table comparisons: PASS
16 reporting view comparisons: PASS
SQL column names and types: PASS
Row-count reconciliation: PASS
Normalized data-value reconciliation: PASS
Business query reconciliation: PASS
Original database protected: PASS

===== STEP 4 COMPLETE =====

Comparison connection closed safely.


In [5]:
# STEP 5: REBUILD DATABASE FROM SAVED MANIFESTS ONLY

from pathlib import Path
import pandas as pd
import duckdb
import re

# ------------------------------------------------------------
# PART 1: IDENTIFY STORED CONFIGURATION FILES
# ------------------------------------------------------------

config_dir = project_root / "config"
processed_dir = project_root / "data" / "processed"

manifest_only_path = (
    project_root
    / "data"
    / "warehouse"
    / "demand_intelligence_manifest_only.duckdb"
)

assert manifest_only_path.resolve() not in {
    database_path.resolve(),
    rebuild_path.resolve()
}

required_config_files = [
    "table_manifest.csv",
    "column_schema_manifest.csv",
    "metadata_column_manifest.csv",
    "reporting_view_manifest.csv",
    "metadata_architecture_catalog.csv",
    "metadata_data_quality_report.csv",
    "metadata_kpi_dictionary.csv"
]

for filename in required_config_files:
    assert (config_dir / filename).is_file(), (
        f"Missing configuration file: {filename}"
    )

print("===== MANIFEST-ONLY REBUILD =====")
print("Configuration files:", len(required_config_files))
print("Output database:", manifest_only_path)
print("Original database connection required: NO")


# ------------------------------------------------------------
# PART 2: LOAD CONFIGURATION MANIFESTS
# ------------------------------------------------------------

table_specs = pd.read_csv(
    config_dir / "table_manifest.csv"
)

column_specs = pd.read_csv(
    config_dir / "column_schema_manifest.csv"
)

metadata_specs = pd.read_csv(
    config_dir / "metadata_column_manifest.csv"
)

saved_views = pd.read_csv(
    config_dir / "reporting_view_manifest.csv"
)

assert len(table_specs) == 37
assert len(saved_views) == 16

metadata_expected_rows = {
    "architecture_catalog": 37,
    "data_quality_report": 37,
    "kpi_dictionary": 8
}

def qi(value):
    return '"' + str(value).replace('"', '""') + '"'


# ------------------------------------------------------------
# PART 3: DEFINE REUSABLE TABLE BUILDER
# ------------------------------------------------------------

def build_manifest_table(
    connection,
    schema,
    table,
    dataframe,
    definitions
):

    definitions = definitions.sort_values(
        "Column_Position"
    )

    expected_columns = definitions[
        "Column_Name"
    ].tolist()

    assert dataframe.columns.tolist() == expected_columns

    expressions = []

    for column in definitions.itertuples(index=False):

        safe_column = qi(column.Column_Name)

        expressions.append(
            f"CAST({safe_column} AS {column.Data_Type}) "
            f"AS {safe_column}"
        )

    select_columns = ", ".join(expressions)

    target = f"{qi(schema)}.{qi(table)}"

    connection.register(
        "manifest_input",
        dataframe
    )

    try:
        connection.execute(
            f"""
            CREATE TABLE {target} AS
            SELECT {select_columns}
            FROM manifest_input
            """
        ).fetchall()

    finally:
        connection.unregister("manifest_input")

    actual_rows = connection.execute(
        f"SELECT COUNT(*) FROM {target}"
    ).fetchone()[0]

    assert actual_rows == len(dataframe)

    actual_columns = connection.execute(
        """
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = ?
          AND table_name = ?
        ORDER BY ordinal_position
        """,
        [schema, table]
    ).fetchall()

    expected_types = list(
        zip(
            definitions["Column_Name"],
            definitions["Data_Type"]
        )
    )

    assert actual_columns == expected_types, (
        f"Column schema mismatch: {target}"
    )

    return actual_rows


# ------------------------------------------------------------
# PART 4: CREATE CLEAN INDEPENDENT DATABASE
# ------------------------------------------------------------

manifest_con = duckdb.connect(
    str(manifest_only_path)
)

transaction_open = False

try:

    manifest_con.execute(
        "BEGIN TRANSACTION"
    ).fetchall()

    transaction_open = True

    # Reset only the manifest-only rebuild database.

    all_schemas = [
        "reporting",
        "metadata",
        "demand",
        "ecommerce",
        "intent",
        "inventory",
        "retail",
        "sensitivity"
    ]

    for schema in all_schemas:

        manifest_con.execute(
            f"DROP SCHEMA IF EXISTS {qi(schema)} CASCADE"
        ).fetchall()

    for schema in all_schemas:

        manifest_con.execute(
            f"CREATE SCHEMA {qi(schema)}"
        ).fetchall()


    # --------------------------------------------------------
    # PART 5: REBUILD THE 37 ANALYTICAL SOURCE TABLES
    # --------------------------------------------------------

    print("\n===== REBUILDING ANALYTICAL TABLES =====")

    source_records = []

    for source in table_specs.itertuples(index=False):

        csv_path = processed_dir / source.File_Name

        assert csv_path.is_file()

        df = pd.read_csv(csv_path)

        assert len(df) == int(source.CSV_Rows)

        definitions = column_specs[
            (column_specs["Schema"] == source.Schema)
            &
            (
                column_specs["Table_Name"]
                == source.Table_Name
            )
        ]

        assert not definitions.empty

        rows = build_manifest_table(
            manifest_con,
            source.Schema,
            source.Table_Name,
            df,
            definitions
        )

        source_records.append({
            "Schema": source.Schema,
            "Table": source.Table_Name,
            "Rows": rows,
            "Status": "PASS"
        })

    assert len(source_records) == 37

    print("Analytical tables rebuilt:", len(source_records))


    # --------------------------------------------------------
    # PART 6: REBUILD METADATA FROM SAVED SNAPSHOTS
    # --------------------------------------------------------

    print("\n===== REBUILDING METADATA =====")

    metadata_records = []

    for table, expected_rows in (
        metadata_expected_rows.items()
    ):

        csv_path = (
            config_dir / f"metadata_{table}.csv"
        )

        # Preserve literal "N/A" metadata values.
        # Empty CSV fields remain missing values.

        df = pd.read_csv(
            csv_path,
            keep_default_na=False,
            na_values=[""]
        )

        assert len(df) == expected_rows

        definitions = (
            metadata_specs[
                metadata_specs["table_name"] == table
            ]
            .rename(columns={
                "ordinal_position": "Column_Position",
                "column_name": "Column_Name",
                "data_type": "Data_Type"
            })
        )

        assert not definitions.empty

        rows = build_manifest_table(
            manifest_con,
            "metadata",
            table,
            df,
            definitions
        )

        metadata_records.append({
            "Table": table,
            "Rows": rows,
            "Status": "PASS"
        })

        print(f"Restored: metadata.{table} ({rows} rows)")


    # --------------------------------------------------------
    # PART 7: REBUILD REPORTING VIEWS
    # --------------------------------------------------------

    print("\n===== REBUILDING REPORTING VIEWS =====")

    view_pattern = re.compile(
        r"^\s*CREATE\s+"
        r"(?:OR\s+REPLACE\s+)?"
        r"(?:TEMPORARY\s+)?"
        r"VIEW\s+"
        r"(?:IF\s+NOT\s+EXISTS\s+)?"
        r".+?\s+AS\s+",
        flags=re.IGNORECASE | re.DOTALL
    )

    pending = {}

    for row in saved_views.itertuples(index=False):

        stored_sql = str(row.sql).strip()

        match = view_pattern.match(stored_sql)

        assert match, (
            f"Cannot parse saved view: {row.view_name}"
        )

        pending[row.view_name] = (
            stored_sql[match.end():]
            .strip()
            .rstrip(";")
        )

    # Resolve dependencies among reporting views.

    for attempt in range(16):

        if not pending:
            break

        remaining = {}
        progress = False

        for view_name, select_sql in pending.items():

            try:
                manifest_con.execute(
                    f"""
                    CREATE OR REPLACE VIEW
                    reporting.{qi(view_name)}
                    AS
                    {select_sql}
                    """
                ).fetchall()

                progress = True

            except duckdb.Error as error:
                remaining[view_name] = (
                    select_sql,
                    str(error)
                )

        if not progress and remaining:

            details = "\n".join(
                f"{name}: {error}"
                for name, (_, error) in remaining.items()
            )

            raise RuntimeError(
                "Unable to restore views:\n" + details
            )

        pending = {
            name: sql
            for name, (sql, _) in remaining.items()
        }

    assert not pending

    print("Reporting views recreated:", len(saved_views))


    # --------------------------------------------------------
    # PART 8: VERIFY COMPLETE DATABASE STRUCTURE
    # --------------------------------------------------------

    table_count = manifest_con.execute(
        """
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema IN (
            'demand',
            'ecommerce',
            'intent',
            'inventory',
            'retail',
            'sensitivity',
            'metadata'
        )
        AND table_type = 'BASE TABLE'
        """
    ).fetchone()[0]

    view_count = manifest_con.execute(
        """
        SELECT COUNT(*)
        FROM information_schema.views
        WHERE table_schema = 'reporting'
        """
    ).fetchone()[0]

    assert table_count == 40
    assert view_count == 16

    # Ensure every reporting view executes.
    for view_name in saved_views["view_name"]:

        manifest_con.execute(
            f"""
            SELECT COUNT(*)
            FROM reporting.{qi(view_name)}
            """
        ).fetchone()


    # --------------------------------------------------------
    # PART 9: VALIDATE KEY BUSINESS METRICS
    # --------------------------------------------------------

    metric_checks = {
        "AirPods Historical Units": (
            """
            SELECT SUM(Units_Sold)
            FROM reporting.v_airpods_weekly_demand
            """,
            15364
        ),

        "Supplier Delay Shortage": (
            """
            SELECT SUM(Shortage_Units)
            FROM reporting.v_inventory_weekly
            WHERE Scenario = 'Supplier Delay'
            """,
            660
        ),

        "No READY Option Cases": (
            """
            SELECT COUNT(*)
            FROM reporting.v_sensitivity_decision_cases
            WHERE Lowest_Ready_Action = 'NO READY OPTION'
              AND Lowest_Ready_Exposure_USD IS NULL
            """,
            72
        )
    }

    metric_records = []

    for name, (query, expected) in metric_checks.items():

        actual = manifest_con.execute(
            query
        ).fetchone()[0]

        assert actual == expected

        metric_records.append({
            "Metric": name,
            "Expected": expected,
            "Actual": actual,
            "Status": "PASS"
        })


    # --------------------------------------------------------
    # PART 10: COMMIT SUCCESSFUL REBUILD
    # --------------------------------------------------------

    manifest_con.execute("COMMIT").fetchall()

    transaction_open = False

    print("\n===== MANIFEST-ONLY REBUILD REPORT =====")

    display(pd.DataFrame(metric_records))

    print("\n===== FINAL REBUILD VALIDATION =====")

    print("Configuration files loaded: 7")
    print("Analytical tables:", len(source_records))
    print("Metadata tables:", len(metadata_records))
    print("Total persistent tables:", table_count)
    print("Reporting views:", view_count)
    print("Business metrics reconciled: 3/3")
    print("Original database connection needed: NO")
    print("Isolated manifest rebuild: PASS")

    print("\n===== STEP 5 COMPLETE =====")

except Exception:

    if transaction_open:
        manifest_con.execute("ROLLBACK").fetchall()

    raise

finally:

    manifest_con.close()

    print("\nManifest database connection closed.")


===== MANIFEST-ONLY REBUILD =====
Configuration files: 7
Output database: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence_manifest_only.duckdb
Original database connection required: NO

===== REBUILDING ANALYTICAL TABLES =====
Analytical tables rebuilt: 37

===== REBUILDING METADATA =====
Restored: metadata.architecture_catalog (37 rows)
Restored: metadata.data_quality_report (37 rows)
Restored: metadata.kpi_dictionary (8 rows)

===== REBUILDING REPORTING VIEWS =====
Reporting views recreated: 16

===== MANIFEST-ONLY REBUILD REPORT =====


,Metric,Expected,Actual,Status
0,AirPods Historical Units,15364,15364.0,PASS
1,Supplier Delay Shortage,660,660.0,PASS
2,No READY Option Cases,72,72.0,PASS



===== FINAL REBUILD VALIDATION =====
Configuration files loaded: 7
Analytical tables: 37
Metadata tables: 3
Total persistent tables: 40
Reporting views: 16
Business metrics reconciled: 3/3
Original database connection needed: NO
Isolated manifest rebuild: PASS

===== STEP 5 COMPLETE =====

Manifest database connection closed.


In [6]:
# STEP 6: VALIDATE INDEPENDENT MANIFEST-ONLY REBUILD

from pathlib import Path
import pandas as pd
import duckdb

# ------------------------------------------------------------
# PART 1: LOAD SAVED OBJECT MANIFESTS
# ------------------------------------------------------------

table_manifest = pd.read_csv(
    config_dir / "table_manifest.csv"
)

view_manifest = pd.read_csv(
    config_dir / "reporting_view_manifest.csv"
)

metadata_tables = [
    "architecture_catalog",
    "data_quality_report",
    "kpi_dictionary"
]

assert len(table_manifest) == 37
assert len(view_manifest) == 16

assert database_path.is_file()
assert manifest_only_path.is_file()

assert (
    database_path.resolve()
    != manifest_only_path.resolve()
)

print("===== INDEPENDENT REBUILD VALIDATION =====")

print("Original database:", database_path.name)
print("Manifest rebuild:", manifest_only_path.name)


# ------------------------------------------------------------
# PART 2: CONSTRUCT OBJECT INVENTORY
# ------------------------------------------------------------

objects = []

for row in table_manifest.itertuples(index=False):

    objects.append({
        "Type": "TABLE",
        "Schema": row.Schema,
        "Name": row.Table_Name
    })

for name in metadata_tables:

    objects.append({
        "Type": "TABLE",
        "Schema": "metadata",
        "Name": name
    })

for name in view_manifest["view_name"]:

    objects.append({
        "Type": "VIEW",
        "Schema": "reporting",
        "Name": name
    })

assert len(objects) == 56

assert len({
    (o["Schema"], o["Name"])
    for o in objects
}) == 56

print("\n===== COMPARISON SCOPE =====")

print("SQL tables:", 40)
print("Reporting views:", 16)
print("Total SQL objects:", len(objects))


# ------------------------------------------------------------
# PART 3: DEFINE SAFE SQL UTILITIES
# ------------------------------------------------------------

def sql_identifier(value):

    return '"' + str(value).replace('"', '""') + '"'


def sql_file_path(value):

    return "'" + str(value).replace("'", "''") + "'"


def qualified_name(database, schema, name):

    return ".".join([
        sql_identifier(database),
        sql_identifier(schema),
        sql_identifier(name)
    ])


# ------------------------------------------------------------
# PART 4: OPEN BOTH DATABASES READ-ONLY
# ------------------------------------------------------------

validation_con = duckdb.connect(":memory:")

try:

    validation_con.execute(
        f"""
        ATTACH {sql_file_path(database_path)}
        AS source_db (READ_ONLY)
        """
    ).fetchall()

    validation_con.execute(
        f"""
        ATTACH {sql_file_path(manifest_only_path)}
        AS manifest_db (READ_ONLY)
        """
    ).fetchall()

    print("\n===== READ-ONLY CONNECTIONS =====")
    print("Original database: CONNECTED")
    print("Manifest database: CONNECTED")


    # --------------------------------------------------------
    # PART 5: DEFINE SCHEMA AND DATA COMPARISON
    # --------------------------------------------------------

    def get_columns(table_path):

        rows = validation_con.execute(
            f"DESCRIBE SELECT * FROM {table_path}"
        ).fetchall()

        return [
            (row[0], str(row[1]).upper())
            for row in rows
        ]


    def normalized_projection(columns):

        expressions = []

        for name, data_type in columns:

            safe_name = sql_identifier(name)

            # Normalize floating-point values
            # to six decimal places.

            if (
                data_type in (
                    "DOUBLE", "FLOAT", "REAL"
                )
                or data_type.startswith("FLOAT")
            ):

                expressions.append(
                    f"ROUND({safe_name}, 6)"
                )

            else:

                expressions.append(safe_name)

        return ", ".join(expressions)


    # --------------------------------------------------------
    # PART 6: COMPARE ALL 56 DATABASE OBJECTS
    # --------------------------------------------------------

    comparison_records = []

    for obj in objects:

        source_name = qualified_name(
            "source_db",
            obj["Schema"],
            obj["Name"]
        )

        rebuilt_name = qualified_name(
            "manifest_db",
            obj["Schema"],
            obj["Name"]
        )

        source_columns = get_columns(source_name)
        rebuilt_columns = get_columns(rebuilt_name)

        schema_match = (
            source_columns == rebuilt_columns
        )

        source_rows = validation_con.execute(
            f"SELECT COUNT(*) FROM {source_name}"
        ).fetchone()[0]

        rebuilt_rows = validation_con.execute(
            f"SELECT COUNT(*) FROM {rebuilt_name}"
        ).fetchone()[0]

        source_only = None
        rebuilt_only = None

        if schema_match:

            projection = normalized_projection(
                source_columns
            )

            # EXCEPT ALL compares row values and
            # preserves duplicate-row multiplicities.

            source_only = validation_con.execute(
                f"""
                SELECT COUNT(*)
                FROM (
                    SELECT {projection}
                    FROM {source_name}

                    EXCEPT ALL

                    SELECT {projection}
                    FROM {rebuilt_name}
                ) AS differences
                """
            ).fetchone()[0]

            rebuilt_only = validation_con.execute(
                f"""
                SELECT COUNT(*)
                FROM (
                    SELECT {projection}
                    FROM {rebuilt_name}

                    EXCEPT ALL

                    SELECT {projection}
                    FROM {source_name}
                ) AS differences
                """
            ).fetchone()[0]

        passed = (
            schema_match
            and source_rows == rebuilt_rows
            and source_only == 0
            and rebuilt_only == 0
        )

        comparison_records.append({
            "Type": obj["Type"],
            "Schema": obj["Schema"],
            "Object": obj["Name"],
            "Original_Rows": source_rows,
            "Rebuilt_Rows": rebuilt_rows,
            "Schema_Match": schema_match,
            "Original_Only": source_only,
            "Rebuilt_Only": rebuilt_only,
            "Status": (
                "PASS" if passed else "REVIEW"
            )
        })

        print(
            f"{'PASS' if passed else 'REVIEW'}: "
            f"{obj['Schema']}.{obj['Name']}"
        )


    # --------------------------------------------------------
    # PART 7: SUMMARIZE VALIDATION RESULTS
    # --------------------------------------------------------

    independent_parity = pd.DataFrame(
        comparison_records
    )

    parity_summary = (
        independent_parity
        .groupby(["Type", "Status"])
        .size()
        .reset_index(name="Objects")
    )

    print("\n===== INDEPENDENT REBUILD PARITY SUMMARY =====")

    display(parity_summary)

    failures = independent_parity[
        independent_parity["Status"] != "PASS"
    ]

    print("\n===== OBJECTS REQUIRING REVIEW =====")

    if failures.empty:
        print("None. All objects passed.")
    else:
        display(failures)


    # --------------------------------------------------------
    # PART 8: VALIDATE BUSINESS METRICS
    # --------------------------------------------------------

    metric_queries = {
        "AirPods Historical Units": (
            """
            SELECT SUM(Units_Sold)
            FROM {db}.reporting.v_airpods_weekly_demand
            """,
            15364
        ),
        "Supplier Delay Shortages": (
            """
            SELECT SUM(Shortage_Units)
            FROM {db}.reporting.v_inventory_weekly
            WHERE Scenario = 'Supplier Delay'
            """,
            660
        ),
        "No READY Option Cases": (
            """
            SELECT COUNT(*)
            FROM {db}.reporting.v_sensitivity_decision_cases
            WHERE Lowest_Ready_Action = 'NO READY OPTION'
              AND Lowest_Ready_Exposure_USD IS NULL
            """,
            72
        )
    }

    metric_results = []

    for metric, (template, expected) in (
        metric_queries.items()
    ):

        source_value = validation_con.execute(
            template.format(
                db=sql_identifier("source_db")
            )
        ).fetchone()[0]

        rebuilt_value = validation_con.execute(
            template.format(
                db=sql_identifier("manifest_db")
            )
        ).fetchone()[0]

        passed = (
            source_value == rebuilt_value
            and rebuilt_value == expected
        )

        metric_results.append({
            "Metric": metric,
            "Original": source_value,
            "Manifest_Rebuild": rebuilt_value,
            "Expected": expected,
            "Status": (
                "PASS" if passed else "REVIEW"
            )
        })

    metric_report = pd.DataFrame(metric_results)

    print("\n===== BUSINESS KPI VALIDATION =====")

    display(metric_report)


    # --------------------------------------------------------
    # PART 9: FINAL VALIDATION
    # --------------------------------------------------------

    assert len(independent_parity) == 56

    assert len(
        independent_parity[
            independent_parity["Type"] == "TABLE"
        ]
    ) == 40

    assert len(
        independent_parity[
            independent_parity["Type"] == "VIEW"
        ]
    ) == 16

    assert failures.empty, (
        "Database parity requires review."
    )

    assert (
        metric_report["Status"] == "PASS"
    ).all()

    print("\n===== FINAL INDEPENDENT REBUILD VALIDATION =====")

    print("40 analytical and metadata tables: PASS")
    print("16 SQL reporting views: PASS")
    print("Column schema comparison: PASS")
    print("Row-count comparison: PASS")
    print("Normalized record-value comparison: PASS")
    print("Business KPI reconciliation: PASS")
    print("Both databases protected: PASS")

    print("\n===== STEP 6 COMPLETE =====")

finally:

    validation_con.close()

    print("\nValidation connections closed safely.")


===== INDEPENDENT REBUILD VALIDATION =====
Original database: demand_intelligence.duckdb
Manifest rebuild: demand_intelligence_manifest_only.duckdb

===== COMPARISON SCOPE =====
SQL tables: 40
Reporting views: 16
Total SQL objects: 56

===== READ-ONLY CONNECTIONS =====
Original database: CONNECTED
Manifest database: CONNECTED
PASS: demand.airpods_weekly_2019
PASS: demand.forecast_lead_time_baseline_2019
PASS: demand.forecast_multihorizon_performance_2019
PASS: demand.forecast_multihorizon_predictions_2019
PASS: demand.forecast_stability_summary_2019
PASS: demand.forecast_uncertainty_summary_2019
PASS: demand.model_performance_rolling_2019
PASS: demand.model_predictions_rolling_2019
PASS: ecommerce.apple_headphones_daily_2019
PASS: ecommerce.apple_headphones_weekly_2019
PASS: ecommerce.demand_sensing_holdout_predictions_2019
PASS: ecommerce.demand_sensing_model_comparison_2019
PASS: intent.purchase_intent_confusion_summary
PASS: intent.purchase_intent_model_leaderboard
PASS: inventory.i

,Type,Status,Objects
0,TABLE,PASS,40
1,VIEW,PASS,16



===== OBJECTS REQUIRING REVIEW =====
None. All objects passed.

===== BUSINESS KPI VALIDATION =====


,Metric,Original,Manifest_Rebuild,Expected,Status
0,AirPods Historical Units,15364.0,15364.0,15364,PASS
1,Supplier Delay Shortages,660.0,660.0,660,PASS
2,No READY Option Cases,72.0,72.0,72,PASS



===== FINAL INDEPENDENT REBUILD VALIDATION =====
40 analytical and metadata tables: PASS
16 SQL reporting views: PASS
Column schema comparison: PASS
Row-count comparison: PASS
Normalized record-value comparison: PASS
Business KPI reconciliation: PASS
Both databases protected: PASS

===== STEP 6 COMPLETE =====

Validation connections closed safely.



# Reproducible Demand Intelligence Pipeline — Final Findings

## 1. Project Objective

Develop and validate a reproducible analytical warehouse
supporting demand forecasting, purchase-intent analytics,
inventory planning, replenishment decisions, and
sensitivity analysis.

The primary objective was to reconstruct the database
from processed CSV datasets and saved configuration
manifests without depending on the original database.

## 2. Existing Architecture Audit

Audited the existing analytical warehouse:

- 37 processed CSV datasets
- 40 persistent SQL tables
- 16 reporting views
- 37 architecture catalog entries
- 37 data-quality catalog entries

All 37 source datasets reconciled with their respective
SQL tables and metadata row counts.

## 3. Database Reconstruction

Created a separate DuckDB rebuild candidate.

Captured:
- Source-to-table manifest
- Column schema manifest
- Metadata table snapshots
- Metadata column schema definitions
- Reporting view SQL definitions

Reconstructed 37 analytical source tables.

Restored three metadata tables and 16 reporting views.

The original warehouse remained unchanged.

## 4. First Database Parity Validation

Compared the original warehouse with the first rebuild.

Validation scope:
- 40 SQL tables
- 16 reporting views
- 56 total SQL objects

Verified:
- Column names and types
- Record counts
- Normalized record values
- Duplicate-row multiplicities
- Three selected business KPIs

All 56 objects passed.

Floating-point values were compared after rounding
to six decimal places.

## 5. Manifest-Only Rebuild

Created an additional independent DuckDB database
using seven saved configuration files and the 37
processed analytical CSV datasets.

Reconstructed:
- 37 analytical tables
- 3 metadata tables
- 16 SQL reporting views

The rebuild did not require a connection to the
original database.

## 6. Independent Rebuild Validation

Compared the manifest-only database to the original
warehouse across all 56 SQL objects.

Results:
- 40 of 40 tables passed
- 16 of 16 reporting views passed
- Zero objects required review
- All three benchmark business queries passed

The comparison was independent of record ordering
and preserved duplicate-row multiplicities.

## 7. Business KPI Reconciliation

| KPI | Original | Manifest Rebuild |
|---|---:|---:|
| Historical AirPods Units | 15,364 | 15,364 |
| Supplier-Delay Shortage Units | 660 | 660 |
| Cases With No READY Option | 72 | 72 |

All three checks passed.

The shortage and readiness results originate from
hypothetical planning scenarios.

## 8. Technical Deliverables

Implemented:
- Data architecture audit
- SQL table and column schema manifests
- Metadata and reporting view configuration snapshots
- Isolated DuckDB reconstruction
- Manifest-driven database rebuild
- SQL schema and value-parity checks
- Automated business KPI validation

Core technologies:
- Python
- Pandas
- DuckDB
- SQL
- Jupyter Notebook
- Git / GitHub

## 9. Business Implications

The analytical warehouse can be reconstructed
consistently from its saved processed-data inputs.

This improves:
- Analytical traceability
- Reproducibility
- Reporting consistency
- Data validation
- Confidence in downstream dashboard inputs

It also reduces dependence on manually rebuilding
individual SQL tables across earlier notebooks.

## 10. Limitations

- The pipeline starts from processed CSV datasets,
  not the original raw Kaggle files.
- Configuration snapshots were initially derived
  from the validated original database.
- Floating-point parity was evaluated to six
  decimal places, not byte-for-byte equality.
- Metadata currently reflects the saved source
  snapshots, rather than automated source discovery.
- This is not a scheduled production ETL pipeline.
- No external Helium 10 data has been integrated yet.
- Power BI dashboard development remains incomplete.
- Historical forecasting and hypothetical inventory
  scenarios do not represent measured business impact.

## 11. Next Development Phase

1. Assess Helium 10 external demand-signal feasibility.
2. Identify compatible product and time-series fields.
3. Integrate usable external demand indicators.
4. Design and build Power BI executive dashboards.
5. Validate dashboard metrics against SQL sources.
6. Improve end-to-end documentation and presentation.
7. Publish the final GitHub and website case study.
